# Baseline Models

A baseline shows what a model can reach **without learning anything useful**. Every real model must beat it.

- **Always Home win:** high accuracy because Home wins are the most common result, but it never predicts a draw or an away win.
- **Random by class share:** guesses according to the class percentages.
- **Bookmaker favourite (reference):** the outcome with the highest implied probability.

## 1. Load the final tables

In [2]:
import os
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Define path to your project folder
project_path = '/content/drive/MyDrive/Y3S1/Assignment/Models/'

# Change the current working directory to the project folder
if os.path.exists(project_path):
    os.chdir(project_path)
    print(f"Successfully changed working directory to: {os.getcwd()}")
else:
    print(f"Error: Path '{project_path}' does not exist. Please double-check your Google Drive folder structure.")

Mounted at /content/drive
Successfully changed working directory to: /content/drive/MyDrive/Y3S1/Assignment/Models


In [3]:
import pandas as pd, numpy as np, json, warnings
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.metrics import (accuracy_score, f1_score, log_loss, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay)
warnings.filterwarnings("ignore")

# Final tables produced by the feature-engineering notebook (upload the 5 files to the same folder)
train      = pd.read_csv("train_final.csv", parse_dates=["date"]).sort_values(["date", "match_api_id"])
test       = pd.read_csv("test_final.csv", parse_dates=["date"])
train_odds = pd.read_csv("train_final_odds.csv", parse_dates=["date"]).sort_values(["date", "match_api_id"])
test_odds  = pd.read_csv("test_final_odds.csv", parse_dates=["date"])
feats = json.load(open("final_feature_sets.json"))

# Two experiments: our own features only, and our features + bookmaker odds
experiments = {
    "No odds":   (train,      test,      feats["features_no_odds"]),
    "With odds": (train_odds, test_odds, feats["features_with_odds"]),
}
class_names = ["Away win", "Draw", "Home win"]      # target: 0 = A, 1 = D, 2 = H
for name, (tr, te, cols) in experiments.items():
    print(f"{name:10s} train {tr.shape[0]:6d} | test {te.shape[0]:5d} | features {len(cols)}")

No odds    train  19328 | test  6651 | features 16
With odds  train  16783 | test  5809 | features 18


## 2. Evaluation helper

In [4]:
def evaluate(model, X, y, label, exp):
    """Test-set metrics. Macro F1 is the main metric because draws are the hardest class."""
    pred = model.predict(X)
    proba = model.predict_proba(X) if hasattr(model, "predict_proba") else None
    report = classification_report(y, pred, output_dict=True, zero_division=0)
    return {"model": label, "features": exp,
            "accuracy": round(accuracy_score(y, pred), 4),
            "macro_F1": round(f1_score(y, pred, average="macro"), 4),
            "weighted_F1": round(f1_score(y, pred, average="weighted"), 4),
            "log_loss": round(log_loss(y, proba), 4) if proba is not None else np.nan,
            "draw_recall": round(report["1"]["recall"], 4)}

def show_confusion(model, X, y, title):
    ConfusionMatrixDisplay.from_predictions(y, model.predict(X), display_labels=class_names, cmap="Blues", colorbar=False)
    plt.title(title)
    plt.show()

## 3. Baseline results on the test seasons

In [5]:
# Baseline 1: always predict the most common class (Home win) | Baseline 2: random guess following class shares
from sklearn.dummy import DummyClassifier
results = []
for exp, (tr, te, cols) in experiments.items():
    for strategy, label in [("most_frequent", "Baseline: always Home win"), ("stratified", "Baseline: random by class share")]:
        dummy = DummyClassifier(strategy=strategy, random_state=42).fit(tr[cols], tr["target"])
        results.append(evaluate(dummy, te[cols], te["target"], label, exp))
    # Bookmaker favourite (only where odds exist): lowest odds = highest implied probability
    if exp == "With odds":
        p = te[["prob_H", "prob_D"]].copy(); p["prob_A"] = 1 - p.prob_H - p.prob_D
        fav = p[["prob_A", "prob_D", "prob_H"]].values.argmax(axis=1)
        results.append({"model": "Reference: bookmaker favourite", "features": exp,
                        "accuracy": round(accuracy_score(te["target"], fav), 4),
                        "macro_F1": round(f1_score(te["target"], fav, average="macro"), 4),
                        "weighted_F1": round(f1_score(te["target"], fav, average="weighted"), 4),
                        "log_loss": np.nan, "draw_recall": round(((fav == 1) & (te["target"] == 1)).sum() / (te["target"] == 1).sum(), 4)})
pd.DataFrame(results)

,model,features,accuracy,macro_F1,weighted_F1,log_loss,draw_recall
0,Baseline: always Home win,No odds,0.4440,0.2050,0.2730,20.0405,0.0000
1,Baseline: random by class share,No odds,0.3500,0.3270,0.3476,23.4276,0.2651
2,Baseline: always Home win,With odds,0.4460,0.2056,0.2752,19.9670,0.0000
3,Baseline: random by class share,With odds,0.3617,0.3374,0.3593,23.0074,0.2709
4,Reference: bookmaker favourite,With odds,0.5269,0.3870,0.4443,NaN,0.0000


## 4. Save the results

In [6]:
baseline_table = pd.DataFrame(results)
baseline_table.to_csv("results_baseline.csv", index=False)
baseline_table

,model,features,accuracy,macro_F1,weighted_F1,log_loss,draw_recall
0,Baseline: always Home win,No odds,0.4440,0.2050,0.2730,20.0405,0.0000
1,Baseline: random by class share,No odds,0.3500,0.3270,0.3476,23.4276,0.2651
2,Baseline: always Home win,With odds,0.4460,0.2056,0.2752,19.9670,0.0000
3,Baseline: random by class share,With odds,0.3617,0.3374,0.3593,23.0074,0.2709
4,Reference: bookmaker favourite,With odds,0.5269,0.3870,0.4443,NaN,0.0000


## 5. Interpretation

Note how **accuracy can look acceptable while macro F1 is low**: always predicting Home gives about 44% accuracy but a macro F1 near 0.2, because Away and Draw are never predicted. This is why macro F1 and draw recall are used to compare models.